In [ ]:
import sys
import os
sys.path
os.listdir()
os.chdir(os.getcwd().replace("\\","/").replace("/notebooks",""))
sys.path.append("src")

In [ ]:
from src.envConfig import EnvConfig

In [ ]:
EnvConfig()

In [ ]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [ ]:
spark = SparkSession.builder \
    .appName("ExcelComPandasSpark4") \
    .config("spark.sql.ansi.enabled", "false") \
    .config("spark.api.python.worker.connection.timeout", "200") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")\
    .getOrCreate()

In [ ]:
p_df = pd.read_excel("", na_values=["-"])

In [ ]:
df = spark.createDataFrame(p_df)
df.show()

In [ ]:
where = (
    df
    .filter(F.col("Movimentação").isin("Rendimento", "Juros Sobre Capital Próprio"))
)

In [ ]:
select = (
    where
    .select(
        F.col("Data"),
        F.col("Produto"),
        F.col("Preço unitário"),
        F.col("Valor da Operação")
    )
) 

In [ ]:
select = select.withColumn("Codigo Produto", F.substring(F.col("Produto"), 0, F.locate("-", F.col("Produto"))-2))
select = select.withColumn("Produto", F.substring(F.col("Produto"), F.locate("-", F.col("Produto"))+2, 100))



In [ ]:
select.show(truncate=False)

In [ ]:
from pyspark.sql.functions import date_format, to_date
date = (select.withColumn("Data", to_date("Data", "dd/MM/yyyy")))

In [ ]:
order = (
    date
    .orderBy(F.desc("Codigo Produto"), F.desc("Data"))
)

In [ ]:
order.show()

In [ ]:
def calculo_dpa(df, num):
    from pyspark.sql.window import Window
    partition = Window.partitionBy("Codigo Produto").orderBy(F.monotonically_increasing_id())
    result = (
        df
            .withColumn("row_num", F.row_number().over(partition))
            .filter(F.col("row_num") <= num)
            .groupby('Codigo Produto', 'Produto')
            .agg(
                F.mean("Preço unitário").alias("DPA")
            )
            .select(
                F.col('Codigo Produto'),
                F.col('Produto'),
                F.col('DPA')
            )
        )
    return result

In [ ]:
def somar_proventos(df, coluna_data):
    result = (
        df
            .withColumn("Ano", F.year(F.col(coluna_data)))
            .groupby('Ano','Codigo Produto', 'Produto')
            .agg(
                F.sum("Valor da Operação").alias("Total recebido")
            )
            .select(
                F.col('Ano'),
                F.col('Codigo Produto'),
                F.col('Produto'),
                F.col('Total recebido')
            )
        )
    return result

In [ ]:
dpa = calculo_dpa(order, 3)

In [ ]:
total = somar_proventos(order, "Data")

In [ ]:
total.show(truncate=False)

In [ ]:
dpa.show(truncate=False)